In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from tqdm import tqdm
import time

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PyTorch: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA GeForce GTX 1050 Ti


In [2]:
df = pd.read_csv('Dataset/combined_dataset.csv')

In [3]:
X_raw = df.drop(columns=["Label"])

df = df.replace([np.inf, -np.inf], np.nan)

df['Flow Bytes/s'] = df['Flow Bytes/s'].fillna(df['Flow Bytes/s'].median())
df['Flow Packets/s'] = df['Flow Packets/s'].fillna(df['Flow Packets/s'].median())

In [4]:
df['Flow Bytes/s'] = df['Flow Bytes/s'].clip(
    lower=df['Flow Bytes/s'].quantile(0.001),
    upper=df['Flow Bytes/s'].quantile(0.999)
)

df['Flow Packets/s'] = df['Flow Packets/s'].clip(
    lower=df['Flow Packets/s'].quantile(0.001),
    upper=df['Flow Packets/s'].quantile(0.999)
)

In [5]:
scaler = StandardScaler()
X = scaler.fit_transform(df.drop(columns=["Label"]))
Y = df["Label"]

In [6]:
df

,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,...,src_subnet_16,src_subnet_24,dst_ip_a,dst_ip_b,dst_ip_c,dst_ip_d,dst_is_internal,dst_subnet_16,dst_subnet_24,Label
0,443.0,54865.0,6.0,3.0,2.0,0.0,12.0,0.0,6.0,6.0,...,26640.0,6820047.0,192.0,168.0,10.0,5.0,1.0,49320.0,12625930.0,0.0
1,80.0,55054.0,6.0,109.0,1.0,1.0,6.0,6.0,6.0,6.0,...,26640.0,6819868.0,192.0,168.0,10.0,5.0,1.0,49320.0,12625930.0,0.0
2,80.0,55055.0,6.0,52.0,1.0,1.0,6.0,6.0,6.0,6.0,...,26640.0,6819868.0,192.0,168.0,10.0,5.0,1.0,49320.0,12625930.0,0.0
3,443.0,46236.0,6.0,34.0,1.0,1.0,6.0,6.0,6.0,6.0,...,26641.0,6820337.0,192.0,168.0,10.0,16.0,1.0,49320.0,12625930.0,0.0
4,443.0,54863.0,6.0,3.0,2.0,0.0,12.0,0.0,6.0,6.0,...,26643.0,6820804.0,192.0,168.0,10.0,5.0,1.0,49320.0,12625930.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2577657,51114.0,53.0,17.0,32215.0,4.0,2.0,112.0,152.0,28.0,28.0,...,49320.0,12625930.0,192.0,168.0,10.0,3.0,1.0,49320.0,12625930.0,0.0
2577658,24054.0,53.0,17.0,324.0,2.0,2.0,84.0,362.0,42.0,42.0,...,49320.0,12625930.0,192.0,168.0,10.0,3.0,1.0,49320.0,12625930.0,0.0
2577659,443.0,58030.0,6.0,82.0,2.0,1.0,31.0,6.0,31.0,0.0,...,6096.0,1560739.0,192.0,168.0,10.0,51.0,1.0,49320.0,12625930.0,0.0
2577660,51694.0,53.0,17.0,1048635.0,6.0,2.0,192.0,256.0,32.0,32.0,...,49320.0,12625930.0,192.0,168.0,10.0,3.0,1.0,49320.0,12625930.0,0.0


In [7]:
class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)


    def __len__(self):
        return len(self.X)


    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [22]:
class Model(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )


    def forward(self, x):
        return self.net(x)

In [33]:
y_tensor = torch.tensor(Y, dtype=torch.long)  # обязательно long для bincount
class_counts = torch.bincount(y_tensor)       # теперь работает
class_weights = 1.0 / class_counts.float()    # редкие классы получают больший вес
class_weights = class_weights / class_weights.sum()  # нормируем

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

In [34]:
dataset = MyDataset(X, Y)

train_loader = DataLoader(dataset, batch_size=1024, shuffle=True)

model = Model(input_dim=(df.columns.__len__() - 1), num_classes=10).to(device)
# criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [31]:
print(next(model.parameters()).device)


cuda:0


In [43]:
EPOCHS = 20


for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    start_time = time.time()

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f} - Time: {epoch_time:.2f}s")

Epoch 1/20: 100%|██████████| 2518/2518 [00:37<00:00, 67.41it/s]


Epoch 1/20 - Loss: 0.2389 - Time: 37.36s


Epoch 2/20: 100%|██████████| 2518/2518 [00:37<00:00, 66.34it/s]


Epoch 2/20 - Loss: 0.0716 - Time: 37.96s


Epoch 3/20: 100%|██████████| 2518/2518 [00:37<00:00, 67.27it/s]


...

Epoch 18/20 - Loss: 0.4130 - Time: 33.68s


Epoch 19/20: 100%|██████████| 2518/2518 [00:34<00:00, 73.65it/s]


Epoch 19/20 - Loss: 0.4218 - Time: 34.19s


Epoch 20/20: 100%|██████████| 2518/2518 [00:37<00:00, 67.97it/s]

Epoch 20/20 - Loss: 2.1136 - Time: 37.05s


In [44]:
torch.save(model.state_dict(), 'models/model3.pt')

In [40]:
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
import numpy as np

def evaluate_model(model, val_loader, device):
    model.eval()

    # Для всех данных
    all_labels_all = []
    all_preds_all = []

    # Для атакующих классов (1-9)
    all_labels_attack = []
    all_preds_attack = []
    all_probs_attack = []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)

            outputs = model(x)
            preds = torch.argmax(outputs, dim=1)

            # --- Для всех данных ---
            all_labels_all.extend(y.cpu().numpy())
            all_preds_all.extend(preds.cpu().numpy())

            # --- Для атакующих классов ---
            mask = y != 0  # оставляем только атаки
            if mask.sum() > 0:
                x_attack = x[mask]
                y_attack = y[mask]

                outputs_attack = model(x_attack)
                preds_attack = torch.argmax(outputs_attack, dim=1)

                all_labels_attack.extend(y_attack.cpu().numpy())
                all_preds_attack.extend(preds_attack.cpu().numpy())

                # Вероятности для PR/ROC AUC (классы 1-9)
                probs_attack = torch.softmax(outputs_attack, dim=1)[:, 1:]  # только атаки
                all_probs_attack.append(probs_attack.cpu().numpy())

    # --- Метрики на всех данных ---
    print("=== Metrics on ALL data (0-9 classes) ===")
    all_class_names = [f'Class {i}' for i in range(10)]
    print(classification_report(all_labels_all, all_preds_all,
                                target_names=all_class_names, digits=4))

    # --- Метрики только для атакующих классов ---
    print("=== Metrics on ATTACK classes (1-9) ===")
    if len(all_labels_attack) > 0:
        attack_class_names = [f'Class {i}' for i in range(1, 10)]
        attack_labels = list(range(1, 10))
        print(classification_report(all_labels_attack, all_preds_attack,
                                    labels=attack_labels,
                                    target_names=attack_class_names, digits=4))

        # --- PR AUC / ROC AUC для атак ---
        all_probs_attack = np.vstack(all_probs_attack)
        # Смещаем метки атак с 1..9 → 0..8
        all_labels_attack_adj = np.array(all_labels_attack) - 1

        pr_auc = average_precision_score(np.eye(9)[all_labels_attack_adj], all_probs_attack, average='macro')
        roc_auc = roc_auc_score(np.eye(9)[all_labels_attack_adj], all_probs_attack, average='macro')
        print(f"PR AUC (attacks): {pr_auc:.4f}, ROC AUC (attacks): {roc_auc:.4f}")
    else:
        print("No attack samples in validation set.")


evaluate_model(model, train_loader, device)

=== Metrics on ALL data (0-9 classes) ===
              precision    recall  f1-score   support

     Class 0     1.0000    0.9950    0.9975   2272688
     Class 1     0.4243    0.9985    0.5956      1966
     Class 2     0.9995    0.9999    0.9997    128027
     Class 3     0.9967    1.0000    0.9984      7938
     Class 4     0.0043    1.0000    0.0085        36
     Class 5     0.9993    1.0000    0.9996    158930
     Class 6     0.9963    1.0000    0.9981      5897
     Class 7     0.9748    1.0000    0.9872      1507
     Class 8     0.6364    1.0000    0.7778        21
     Class 9     0.9543    0.9939    0.9737       652

    accuracy                         0.9956   2577662
   macro avg     0.7986    0.9987    0.8336   2577662
weighted avg     0.9994    0.9956    0.9974   2577662

=== Metrics on ATTACK classes (1-9) ===
              precision    recall  f1-score   support

     Class 1     1.0000    0.9985    0.9992      1966
     Class 2     1.0000    0.9999    0.9999    128